In [7]:
"""
forecast_returns_by_category.py
Phân tích và dự báo hàng hoàn theo Product_Category với biểu đồ theo thời gian
Dataset: https://www.kaggle.com/datasets/sayalikhot21/synthetic-dataset-for-e-commerce-return-analysis
"""

import os
import pandas as pd
import numpy as np
from prophet import Prophet
from sklearn.metrics import mean_absolute_error, mean_squared_error
import matplotlib.pyplot as plt
import seaborn as sns
import math
from datetime import datetime, timedelta

# --- Cấu hình ---
INPUT_FILE = "ecommerce_returns_synthetic_data.csv"
OUTPUT_DIR = "output_analysis"
FREQ = "D"  # 'D' ngày, 'W' tuần, 'M' tháng

# Tạo thư mục output
os.makedirs(OUTPUT_DIR, exist_ok=True)

# --- Style cho biểu đồ ---
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# --- Load & chuẩn bị dữ liệu ---
def load_data(path=INPUT_FILE):
    """Load và làm sạch dữ liệu"""
    df = pd.read_csv(path)
    
    # Kiểm tra các cột cần thiết
    required_cols = ["Return_Date", "Return_Status", "Product_Category", "Order_ID"]
    missing = [col for col in required_cols if col not in df.columns]
    if missing:
        raise ValueError(f"Thiếu các cột: {missing}")
    
    # Chuyển đổi datetime
    df["Return_Date"] = pd.to_datetime(df["Return_Date"], errors="coerce")
    if "Order_Date" in df.columns:
        df["Order_Date"] = pd.to_datetime(df["Order_Date"], errors="coerce")
    
    # Lọc chỉ lấy hàng đã hoàn
    returns = df[df["Return_Status"].astype(str).str.lower() == "returned"].copy()
    
    if returns.empty:
        print("⚠️  CẢNH BÁO: Không có dữ liệu hàng hoàn!")
        return pd.DataFrame()
    
    print(f"✓ Loaded {len(returns):,} records hàng hoàn")
    print(f"  Khoảng thời gian: {returns['Return_Date'].min().date()} đến {returns['Return_Date'].max().date()}")
    print(f"  Số loại sản phẩm: {returns['Product_Category'].nunique()}")
    
    return returns

# --- 1. Biểu đồ tổng quan theo thời gian ---
def plot_overall_timeline(returns_df):
    """Biểu đồ xu hướng tổng thể hàng hoàn theo thời gian"""
    
    # Tổng hợp theo ngày
    daily = returns_df.groupby("Return_Date").size().reset_index(name="count")
    
    # Tính moving average
    daily["MA_7"] = daily["count"].rolling(window=7, center=True).mean()
    daily["MA_30"] = daily["count"].rolling(window=30, center=True).mean()
    
    fig, axes = plt.subplots(2, 1, figsize=(16, 10))
    
    # Subplot 1: Dữ liệu thô + Moving Average
    ax1 = axes[0]
    ax1.plot(daily["Return_Date"], daily["count"], alpha=0.3, label="Số lượng hàng hoàn (ngày)", linewidth=1)
    ax1.plot(daily["Return_Date"], daily["MA_7"], label="MA 7 ngày", linewidth=2)
    ax1.plot(daily["Return_Date"], daily["MA_30"], label="MA 30 ngày", linewidth=2, linestyle="--")
    ax1.set_ylabel("Số lượng hàng hoàn", fontsize=12)
    ax1.set_title("Xu hướng Hàng Hoàn Theo Thời Gian", fontsize=14, fontweight="bold")
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Subplot 2: Theo tuần
    weekly = returns_df.groupby(pd.Grouper(key="Return_Date", freq="W")).size().reset_index(name="count")
    ax2 = axes[1]
    ax2.bar(weekly["Return_Date"], weekly["count"], width=5, alpha=0.7, color="steelblue")
    ax2.set_xlabel("Thời gian", fontsize=12)
    ax2.set_ylabel("Số lượng hàng hoàn", fontsize=12)
    ax2.set_title("Số Lượng Hàng Hoàn Theo Tuần", fontsize=14, fontweight="bold")
    ax2.grid(True, alpha=0.3, axis="y")
    
    plt.tight_layout()
    filepath = os.path.join(OUTPUT_DIR, "01_overall_timeline.png")
    plt.savefig(filepath, dpi=150, bbox_inches="tight")
    print(f"✓ Saved: {filepath}")
    plt.close()

# --- 2. Phân tích theo Product_Category ---
def plot_category_breakdown(returns_df):
    """Biểu đồ so sánh các loại sản phẩm"""
    
    category_counts = returns_df["Product_Category"].value_counts()
    
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    # Pie chart
    ax1 = axes[0]
    colors = sns.color_palette("Set3", len(category_counts))
    ax1.pie(category_counts.values, labels=category_counts.index, autopct='%1.1f%%',
            startangle=90, colors=colors, textprops={'fontsize': 10})
    ax1.set_title("Tỷ Lệ Hàng Hoàn Theo Loại Sản Phẩm", fontsize=14, fontweight="bold")
    
    # Bar chart
    ax2 = axes[1]
    category_counts.plot(kind="barh", ax=ax2, color=colors)
    ax2.set_xlabel("Số lượng hàng hoàn", fontsize=12)
    ax2.set_ylabel("Loại sản phẩm", fontsize=12)
    ax2.set_title("Số Lượng Hàng Hoàn Theo Loại Sản Phẩm", fontsize=14, fontweight="bold")
    ax2.grid(True, alpha=0.3, axis="x")
    
    # Thêm giá trị
    for i, v in enumerate(category_counts.values):
        ax2.text(v + max(category_counts)*0.01, i, f'{v:,}', va='center', fontsize=10)
    
    plt.tight_layout()
    filepath = os.path.join(OUTPUT_DIR, "02_category_breakdown.png")
    plt.savefig(filepath, dpi=150, bbox_inches="tight")
    print(f"✓ Saved: {filepath}")
    plt.close()

# --- 3. Timeline theo từng Category ---
def plot_category_timeline(returns_df):
    """Xu hướng hàng hoàn theo thời gian cho từng loại sản phẩm"""
    
    categories = returns_df["Product_Category"].unique()
    n_cats = len(categories)
    
    fig, axes = plt.subplots(n_cats, 1, figsize=(16, 4*n_cats), sharex=True)
    if n_cats == 1:
        axes = [axes]
    
    for idx, category in enumerate(sorted(categories)):
        cat_data = returns_df[returns_df["Product_Category"] == category]
        daily = cat_data.groupby("Return_Date").size().reset_index(name="count")
        daily["MA_7"] = daily["count"].rolling(window=7, center=True).mean()
        
        ax = axes[idx]
        ax.plot(daily["Return_Date"], daily["count"], alpha=0.4, linewidth=1)
        ax.plot(daily["Return_Date"], daily["MA_7"], linewidth=2.5, label=f"MA 7 ngày")
        ax.set_ylabel("Số lượng", fontsize=11)
        ax.set_title(f"📦 {category}", fontsize=12, fontweight="bold", loc="left")
        ax.legend(loc="upper right")
        ax.grid(True, alpha=0.3)
    
    axes[-1].set_xlabel("Thời gian", fontsize=12)
    fig.suptitle("Xu Hướng Hàng Hoàn Theo Từng Loại Sản Phẩm", fontsize=16, fontweight="bold", y=1.0)
    plt.tight_layout()
    filepath = os.path.join(OUTPUT_DIR, "03_category_timelines.png")
    plt.savefig(filepath, dpi=150, bbox_inches="tight")
    print(f"✓ Saved: {filepath}")
    plt.close()

# --- 4. Heatmap theo thời gian ---
def plot_category_heatmap(returns_df):
    """Heatmap số lượng hàng hoàn theo tuần và loại sản phẩm"""
    
    # Group theo tuần và category
    returns_df["Week"] = returns_df["Return_Date"].dt.to_period("W")
    pivot = returns_df.groupby(["Week", "Product_Category"]).size().unstack(fill_value=0)
    
    # Chỉ hiển thị 26 tuần gần nhất nếu dữ liệu quá dài
    if len(pivot) > 26:
        pivot = pivot.iloc[-26:]
    
    fig, ax = plt.subplots(figsize=(16, max(8, len(pivot)*0.3)))
    sns.heatmap(pivot.T, annot=True, fmt="d", cmap="YlOrRd", linewidths=0.5, 
                cbar_kws={'label': 'Số lượng hàng hoàn'}, ax=ax)
    ax.set_xlabel("Tuần", fontsize=12)
    ax.set_ylabel("Loại sản phẩm", fontsize=12)
    ax.set_title("Heatmap Hàng Hoàn Theo Tuần và Loại Sản Phẩm", fontsize=14, fontweight="bold")
    
    # Xoay nhãn x để dễ đọc
    ax.set_xticklabels([str(w) for w in pivot.index], rotation=45, ha="right")
    
    plt.tight_layout()
    filepath = os.path.join(OUTPUT_DIR, "04_category_heatmap.png")
    plt.savefig(filepath, dpi=150, bbox_inches="tight")
    print(f"✓ Saved: {filepath}")
    plt.close()
# Phân tích hàng hoàn theo giới tính và độ tuổi
def plot_demographic_analysis(returns_df):
    """Phân tích hàng hoàn theo giới tính và độ tuổi"""
    
    # Kiểm tra cột
    if "User_Gender" not in returns_df.columns or "User_Age" not in returns_df.columns:
        print("⚠️  Thiếu dữ liệu User_Gender hoặc User_Age — bỏ qua phân tích nhân khẩu học.")
        return
    
    # Làm sạch dữ liệu
    returns_df = returns_df.copy()
    returns_df = returns_df.dropna(subset=["User_Age", "User_Gender"])
    returns_df["User_Gender"] = returns_df["User_Gender"].str.strip().str.title()
    
    # Tạo nhóm tuổi
    bins = [0, 17, 25, 35, 45, 55, 65, 100]
    labels = ["<18", "18-25", "26-35", "36-45", "46-55", "56-65", "65+"]
    returns_df["Age_Group"] = pd.cut(returns_df["User_Age"], bins=bins, labels=labels, right=False)
    
    # 1️⃣ Biểu đồ giới tính
    gender_counts = returns_df["User_Gender"].value_counts().sort_index()
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    colors = sns.color_palette("Set2", len(gender_counts))
    axes[0].pie(gender_counts.values, labels=gender_counts.index, autopct='%1.1f%%',
                startangle=90, colors=colors, textprops={'fontsize': 10})
    axes[0].set_title("Tỷ Lệ Hàng Hoàn Theo Giới Tính", fontsize=14, fontweight="bold")
    
    sns.barplot(x=gender_counts.index, y=gender_counts.values, ax=axes[1], palette=colors)
    axes[1].set_ylabel("Số lượng hàng hoàn", fontsize=12)
    axes[1].set_xlabel("Giới tính", fontsize=12)
    axes[1].set_title("Số Lượng Hàng Hoàn Theo Giới Tính", fontsize=14, fontweight="bold")
    for i, v in enumerate(gender_counts.values):
        axes[1].text(i, v + max(gender_counts)*0.01, f"{v:,}", ha="center", fontsize=10)
    
    plt.tight_layout()
    filepath = os.path.join(OUTPUT_DIR, "04a_gender_analysis.png")
    plt.savefig(filepath, dpi=150, bbox_inches="tight")
    print(f"✓ Saved: {filepath}")
    plt.close()
    
    # 2️⃣ Biểu đồ nhóm tuổi
    age_group_counts = returns_df["Age_Group"].value_counts().sort_index()
    plt.figure(figsize=(10, 6))
    sns.barplot(x=age_group_counts.index, y=age_group_counts.values, palette="coolwarm")
    plt.title("Số Lượng Hàng Hoàn Theo Nhóm Tuổi", fontsize=14, fontweight="bold")
    plt.xlabel("Nhóm tuổi", fontsize=12)
    plt.ylabel("Số lượng hàng hoàn", fontsize=12)
    for i, v in enumerate(age_group_counts.values):
        plt.text(i, v + max(age_group_counts)*0.01, f"{v:,}", ha="center", fontsize=10)
    
    plt.tight_layout()
    filepath = os.path.join(OUTPUT_DIR, "04b_age_group_analysis.png")
    plt.savefig(filepath, dpi=150, bbox_inches="tight")
    print(f"✓ Saved: {filepath}")
    plt.close()
    
    # 3️⃣ Heatmap kết hợp (Age × Gender)
    pivot = returns_df.groupby(["Age_Group", "User_Gender"]).size().unstack(fill_value=0)
    plt.figure(figsize=(10, 6))
    sns.heatmap(pivot, annot=True, fmt="d", cmap="YlGnBu", linewidths=0.5, cbar_kws={'label': 'Số lượng hàng hoàn'})
    plt.title("Phân Bố Hàng Hoàn Theo Tuổi và Giới Tính", fontsize=14, fontweight="bold")
    plt.xlabel("Giới tính", fontsize=12)
    plt.ylabel("Nhóm tuổi", fontsize=12)
    plt.tight_layout()
    filepath = os.path.join(OUTPUT_DIR, "04c_age_gender_heatmap.png")
    plt.savefig(filepath, dpi=150, bbox_inches="tight")
    print(f"✓ Saved: {filepath}")
    plt.close()
# --- 5. Dự báo theo Category với Prophet ---
def forecast_by_category(returns_df, periods=30):
    """Dự báo hàng hoàn cho từng loại sản phẩm"""
    
    categories = sorted(returns_df["Product_Category"].unique())
    forecast_results = {}
    
    fig, axes = plt.subplots(len(categories), 1, figsize=(16, 5*len(categories)))
    if len(categories) == 1:
        axes = [axes]
    
    for idx, category in enumerate(categories):
        print(f"  Đang dự báo cho: {category}...")
        
        # Chuẩn bị dữ liệu
        cat_data = returns_df[returns_df["Product_Category"] == category]
        daily = cat_data.groupby("Return_Date").size().reset_index()
        daily.columns = ["ds", "y"]
        
        # Fill missing dates
        full_range = pd.date_range(daily["ds"].min(), daily["ds"].max(), freq="D")
        daily = pd.DataFrame({"ds": full_range}).merge(daily, on="ds", how="left").fillna(0)
        
        # Train/test split
        test_size = min(14, int(0.2 * len(daily)))
        train = daily.iloc[:-test_size]
        test = daily.iloc[-test_size:]
        
        # Prophet model
        m = Prophet(
            daily_seasonality=True,
            weekly_seasonality=True,
            yearly_seasonality=True,
            seasonality_mode='multiplicative'
        )
        m.fit(train)
        
        # Forecast
        future = m.make_future_dataframe(periods=periods, freq="D")
        forecast = m.predict(future)
        
        # Merge với dữ liệu thực
        result = forecast[["ds", "yhat", "yhat_lower", "yhat_upper"]].merge(
            daily, on="ds", how="left"
        )
        result["category"] = category
        forecast_results[category] = result
        
        # Evaluate
        eval_data = result[result["ds"].isin(test["ds"])].dropna(subset=["y", "yhat"])
        if not eval_data.empty:
            mae = mean_absolute_error(eval_data["y"], eval_data["yhat"])
            try:
                rmse = mean_squared_error(eval_data["y"], eval_data["yhat"], squared=False)
            except:
                rmse = math.sqrt(mean_squared_error(eval_data["y"], eval_data["yhat"]))
            print(f"    MAE: {mae:.2f}, RMSE: {rmse:.2f}")
        
        # Plot
        ax = axes[idx]
        ax.plot(result["ds"], result["yhat"], label="Dự báo", linewidth=2)
        ax.plot(result["ds"], result["y"], label="Dữ liệu thực", 
                alpha=0.7, marker="o", markersize=2, linestyle="-")
        ax.fill_between(result["ds"], result["yhat_lower"], result["yhat_upper"],
                        alpha=0.2, label="Khoảng tin cậy")
        ax.axvline(x=train["ds"].max(), color="red", linestyle="--", 
                   alpha=0.6, label="Train/Test split")
        ax.set_ylabel("Số lượng", fontsize=11)
        ax.set_title(f"📈 Dự báo: {category}", fontsize=12, fontweight="bold", loc="left")
        ax.legend(loc="upper left")
        ax.grid(True, alpha=0.3)
    
    axes[-1].set_xlabel("Thời gian", fontsize=12)
    fig.suptitle(f"Dự Báo Hàng Hoàn {periods} Ngày Tới Theo Loại Sản Phẩm", 
                 fontsize=16, fontweight="bold", y=1.0)
    plt.tight_layout()
    filepath = os.path.join(OUTPUT_DIR, "05_category_forecasts.png")
    plt.savefig(filepath, dpi=150, bbox_inches="tight")
    print(f"✓ Saved: {filepath}")
    plt.close()
    
    # Lưu kết quả forecast
    all_forecasts = pd.concat(forecast_results.values(), ignore_index=True)
    csv_path = os.path.join(OUTPUT_DIR, "forecast_by_category.csv")
    all_forecasts.to_csv(csv_path, index=False)
    print(f"✓ Saved forecast data: {csv_path}")
    
    return forecast_results

# --- 6. So sánh các khoảng thời gian ---
def plot_time_period_comparison(returns_df):
    """So sánh hàng hoàn giữa các khoảng thời gian khác nhau"""
    
    # Thêm các cột thời gian
    returns_df["Year"] = returns_df["Return_Date"].dt.year
    returns_df["Month"] = returns_df["Return_Date"].dt.month
    returns_df["Quarter"] = returns_df["Return_Date"].dt.quarter
    returns_df["DayOfWeek"] = returns_df["Return_Date"].dt.day_name()
    
    fig = plt.figure(figsize=(18, 12))
    gs = fig.add_gridspec(3, 2, hspace=0.3, wspace=0.3)
    
    # 1. Theo tháng
    ax1 = fig.add_subplot(gs[0, 0])
    monthly = returns_df.groupby(["Year", "Month", "Product_Category"]).size().reset_index(name="count")
    pivot_month = monthly.pivot_table(values="count", index=["Year", "Month"], 
                                      columns="Product_Category", aggfunc="sum", fill_value=0)
    pivot_month.plot(kind="bar", stacked=True, ax=ax1)
    ax1.set_title("Hàng Hoàn Theo Tháng", fontsize=12, fontweight="bold")
    ax1.set_xlabel("Năm - Tháng", fontsize=10)
    ax1.set_ylabel("Số lượng", fontsize=10)
    ax1.legend(title="Loại SP", bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
    ax1.tick_params(axis='x', rotation=45)
    
    # 2. Theo quý
    ax2 = fig.add_subplot(gs[0, 1])
    quarterly = returns_df.groupby(["Year", "Quarter", "Product_Category"]).size().reset_index(name="count")
    pivot_quarter = quarterly.pivot_table(values="count", index=["Year", "Quarter"],
                                          columns="Product_Category", aggfunc="sum", fill_value=0)
    pivot_quarter.plot(kind="bar", ax=ax2)
    ax2.set_title("Hàng Hoàn Theo Quý", fontsize=12, fontweight="bold")
    ax2.set_xlabel("Năm - Quý", fontsize=10)
    ax2.set_ylabel("Số lượng", fontsize=10)
    ax2.legend(title="Loại SP", bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
    ax2.tick_params(axis='x', rotation=45)
    
    # 3. Theo ngày trong tuần
    ax3 = fig.add_subplot(gs[1, :])
    dow_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
    dow_data = returns_df.groupby(["DayOfWeek", "Product_Category"]).size().reset_index(name="count")
    pivot_dow = dow_data.pivot_table(values="count", index="DayOfWeek",
                                     columns="Product_Category", aggfunc="sum", fill_value=0)
    pivot_dow = pivot_dow.reindex(dow_order)
    pivot_dow.plot(kind="bar", ax=ax3, width=0.8)
    ax3.set_title("Hàng Hoàn Theo Ngày Trong Tuần", fontsize=12, fontweight="bold")
    ax3.set_xlabel("Ngày trong tuần", fontsize=10)
    ax3.set_ylabel("Số lượng", fontsize=10)
    ax3.legend(title="Loại SP", ncol=3, fontsize=9)
    ax3.tick_params(axis='x', rotation=45)
    ax3.grid(True, alpha=0.3, axis='y')
    
    # 4. Box plot theo category
    ax4 = fig.add_subplot(gs[2, :])
    daily_by_cat = returns_df.groupby(["Return_Date", "Product_Category"]).size().reset_index(name="count")
    sns.boxplot(data=daily_by_cat, x="Product_Category", y="count", ax=ax4)
    ax4.set_title("Phân Bố Số Lượng Hàng Hoàn Hàng Ngày Theo Loại", 
                  fontsize=12, fontweight="bold")
    ax4.set_xlabel("Loại sản phẩm", fontsize=10)
    ax4.set_ylabel("Số lượng hàng hoàn/ngày", fontsize=10)
    ax4.tick_params(axis='x', rotation=45)
    ax4.grid(True, alpha=0.3, axis='y')
    
    plt.suptitle("So Sánh Hàng Hoàn Theo Các Khoảng Thời Gian", 
                 fontsize=16, fontweight="bold", y=0.995)
    filepath = os.path.join(OUTPUT_DIR, "06_time_period_comparison.png")
    plt.savefig(filepath, dpi=150, bbox_inches="tight")
    print(f"✓ Saved: {filepath}")
    plt.close()
    
# --- 7. Biểu đồ kết hợp nhân khẩu học và Product_Category ---
def plot_demographic_by_category(returns_df):
    """Phân tích hàng hoàn theo giới tính, nhóm tuổi và loại sản phẩm"""
    
    # Kiểm tra cột cần thiết
    required_cols = ["User_Gender", "User_Age", "Product_Category"]
    missing = [col for col in required_cols if col not in returns_df.columns]
    if missing:
        print(f"⚠️  Thiếu dữ liệu {missing} — bỏ qua biểu đồ kết hợp nhân khẩu học và sản phẩm.")
        return
    
    df = returns_df.copy().dropna(subset=required_cols)
    df["User_Gender"] = df["User_Gender"].str.strip().str.title()
    
    # Tạo nhóm tuổi
    bins = [0, 17, 25, 35, 45, 55, 65, 100]
    labels = ["<18", "18-25", "26-35", "36-45", "46-55", "56-65", "65+"]
    df["Age_Group"] = pd.cut(df["User_Age"], bins=bins, labels=labels, right=False)
    
    # Pivot: Age × Gender × Product_Category
    pivot = df.groupby(["Age_Group", "User_Gender", "Product_Category"]).size().reset_index(name="count")
    
    # Vẽ heatmap cho từng loại sản phẩm
    categories = df["Product_Category"].unique()
    for category in sorted(categories):
        cat_data = pivot[pivot["Product_Category"] == category]
        heatmap_data = cat_data.pivot(index="Age_Group", columns="User_Gender", values="count").fillna(0)
        
        plt.figure(figsize=(8, 6))
        sns.heatmap(heatmap_data, annot=True, fmt="d", cmap="YlGnBu", linewidths=0.5,
                    cbar_kws={'label': 'Số lượng hàng hoàn'})
        plt.title(f"Phân Bố Hàng Hoàn: {category} theo Tuổi × Giới Tính", fontsize=14, fontweight="bold")
        plt.xlabel("Giới tính", fontsize=12)
        plt.ylabel("Nhóm tuổi", fontsize=12)
        plt.tight_layout()
        
        filepath = os.path.join(OUTPUT_DIR, f"07_demographic_{category.replace(' ', '_')}.png")
        plt.savefig(filepath, dpi=150, bbox_inches="tight")
        print(f"✓ Saved: {filepath}")
        plt.close()

# --- 8. Biểu đồ đường Age × Product_Category ---
def plot_age_category_lines(returns_df):
    """Biểu đồ số lượng hàng hoàn theo nhóm tuổi, các đường là từng sản phẩm"""
    
    required_cols = ["User_Age", "Product_Category"]
    missing = [col for col in required_cols if col not in returns_df.columns]
    if missing:
        print(f"⚠️  Thiếu dữ liệu {missing} — bỏ qua biểu đồ Age × Product_Category.")
        return
    
    df = returns_df.copy().dropna(subset=required_cols)
    
    # Tạo nhóm tuổi
    bins = [0, 17, 25, 35, 45, 55, 65, 100]
    labels = ["<18", "18-25", "26-35", "36-45", "46-55", "56-65", "65+"]
    df["Age_Group"] = pd.cut(df["User_Age"], bins=bins, labels=labels, right=False)
    
    # Group theo Age × Product_Category
    agg = df.groupby(["Age_Group", "Product_Category"]).size().reset_index(name="count")
    
    plt.figure(figsize=(10, 6))
    sns.lineplot(data=agg, x="Age_Group", y="count", hue="Product_Category", marker="o")
    
    plt.title("Số Lượng Hàng Hoàn Theo Nhóm Tuổi (Đường theo Sản Phẩm)", fontsize=14, fontweight="bold")
    plt.xlabel("Nhóm tuổi", fontsize=12)
    plt.ylabel("Số lượng hàng hoàn", fontsize=12)
    plt.grid(True, alpha=0.3)
    plt.legend(title="Loại sản phẩm", fontsize=10)
    
    plt.tight_layout()
    filepath = os.path.join(OUTPUT_DIR, "08_age_category_lines.png")
    plt.savefig(filepath, dpi=150, bbox_inches="tight")
    print(f"✓ Saved: {filepath}")
    plt.close()


# --- Main Pipeline ---
def main():
    print("\n" + "="*60)
    print("  PHÂN TÍCH & DỰ BÁO HÀNG HOÀN THEO LOẠI SẢN PHẨM")
    print("="*60 + "\n")
    
    # Load data
    returns_df = load_data(INPUT_FILE)
    if returns_df.empty:
        print("❌ Không có dữ liệu để xử lý!")
        return
    
    print("\n📊 Đang tạo các biểu đồ phân tích...\n")
    
    # Chạy các phân tích
    plot_overall_timeline(returns_df)
    plot_category_breakdown(returns_df)
    plot_category_timeline(returns_df)
    plot_category_heatmap(returns_df)
    plot_demographic_analysis(returns_df)
    plot_time_period_comparison(returns_df)
    plot_demographic_by_category(returns_df)
    plot_age_category_lines(returns_df)
    
    print("\n🔮 Đang chạy dự báo...\n")
    forecast_by_category(returns_df, periods=30)
    
    print("\n" + "="*60)
    print(f"✅ HOÀN THÀNH! Tất cả kết quả đã lưu trong: {OUTPUT_DIR}/")
    print("="*60 + "\n")
    
    # Tổng kết
    print("📁 Các file đã tạo:")
    for i, filename in enumerate([
        "01_overall_timeline.png",
        "02_category_breakdown.png", 
        "03_category_timelines.png",
        "04_category_heatmap.png",
        "04a_gender_analysis.png",
        "04b_age_group_analysis.png",
        "04c_age_gender_heatmap.png",
        "05_category_forecasts.png",
        "06_time_period_comparison.png",
        "08_gender_&_category.png",
        "09_age_category_lines.png"
        "forecast_by_category.csv"
    ], 1):
        print(f"   {i}. {os.path.join(OUTPUT_DIR, filename)}")

if __name__ == "__main__":
    main()


  PHÂN TÍCH & DỰ BÁO HÀNG HOÀN THEO LOẠI SẢN PHẨM

✓ Loaded 5,052 records hàng hoàn
  Khoảng thời gian: 2023-01-01 đến 2024-12-31
  Số loại sản phẩm: 5

📊 Đang tạo các biểu đồ phân tích...

✓ Saved: output_analysis\01_overall_timeline.png
✓ Saved: output_analysis\02_category_breakdown.png


C:\Users\Admin\AppData\Local\Temp\ipykernel_15620\2319669794.py:155: UserWarning: Glyph 128230 (\N{PACKAGE}) missing from font(s) Arial.
  plt.tight_layout()
C:\Users\Admin\AppData\Local\Temp\ipykernel_15620\2319669794.py:157: UserWarning: Glyph 128230 (\N{PACKAGE}) missing from font(s) Arial.
  plt.savefig(filepath, dpi=150, bbox_inches="tight")


✓ Saved: output_analysis\03_category_timelines.png
✓ Saved: output_analysis\04_category_heatmap.png


C:\Users\Admin\AppData\Local\Temp\ipykernel_15620\2319669794.py:216: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x=gender_counts.index, y=gender_counts.values, ax=axes[1], palette=colors)
C:\Users\Admin\AppData\Local\Temp\ipykernel_15620\2319669794.py:232: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x=age_group_counts.index, y=age_group_counts.values, palette="coolwarm")


✓ Saved: output_analysis\04a_gender_analysis.png
✓ Saved: output_analysis\04b_age_group_analysis.png


...

    MAE: 0.72, RMSE: 0.86
  Đang dự báo cho: Toys...


C:\Users\Admin\AppData\Local\Temp\ipykernel_15620\2319669794.py:332: UserWarning: Glyph 128200 (\N{CHART WITH UPWARDS TREND}) missing from font(s) Arial.
  plt.tight_layout()


    MAE: 1.59, RMSE: 2.15


C:\Users\Admin\AppData\Local\Temp\ipykernel_15620\2319669794.py:334: UserWarning: Glyph 128200 (\N{CHART WITH UPWARDS TREND}) missing from font(s) Arial.
  plt.savefig(filepath, dpi=150, bbox_inches="tight")


✓ Saved: output_analysis\05_category_forecasts.png
✓ Saved forecast data: output_analysis\forecast_by_category.csv

✅ HOÀN THÀNH! Tất cả kết quả đã lưu trong: output_analysis/

📁 Các file đã tạo:
   1. output_analysis\01_overall_timeline.png
   2. output_analysis\02_category_breakdown.png
   3. output_analysis\03_category_timelines.png
   4. output_analysis\04_category_heatmap.png
   5. output_analysis\04a_gender_analysis.png
   6. output_analysis\04b_age_group_analysis.png
   7. output_analysis\04c_age_gender_heatmap.png
   8. output_analysis\05_category_forecasts.png
   9. output_analysis\06_time_period_comparison.png
   10. output_analysis\08_gender_&_category.png
   11. output_analysis\09_age_category_lines.pngforecast_by_category.csv


In [4]:
import sklearn, inspect
from sklearn.metrics import mean_squared_error
print(sklearn.__version__)
import inspect
print(inspect.signature(mean_squared_error))

1.5.1
(y_true, y_pred, *, sample_weight=None, multioutput='uniform_average', squared='deprecated')


In [2]:
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import matplotlib.pyplot as plt
from forecast_returns_fixed import forecast_returns

# --- Tiêu đề cho app ---
display(HTML("<h2 style='text-align:center; color:#2E86C1;'>📦 Dự báo số lượng hàng hoàn</h2>"))

# === Các widget điều khiển ===
file_input = widgets.Text(
    value="ecommerce_returns_synthetic_data.csv",
    description="Tên file CSV:",
    style={'description_width': 'initial'},
    layout=widgets.Layout(width="450px")
)
freq_dropdown = widgets.Dropdown(
    options=[("Ngày", "D"), ("Tuần", "W"), ("Tháng", "M")],
    value="D",
    description="Tần suất:",
    style={'description_width': 'initial'},
    layout=widgets.Layout(width="300px")
)
period_slider = widgets.IntSlider(
    value=30, min=7, max=180, step=1,
    description="Dự báo (ngày):",
    style={'description_width': 'initial'},
    continuous_update=False,
    layout=widgets.Layout(width="450px")
)
run_button = widgets.Button(
    description="🚀 Chạy dự báo",
    button_style="success",
    layout=widgets.Layout(width="200px", align_self="center")
)

output_area = widgets.Output()

# === Hàm chạy khi bấm nút ===
def on_run_clicked(b):
    with output_area:
        clear_output()
        print("🔄 Đang chạy dự báo...")
        try:
            forecast_df, model = forecast_returns(
                input_file=file_input.value,
                freq=freq_dropdown.value,
                periods=period_slider.value
            )

            plt.figure(figsize=(12,5))
            plt.plot(forecast_df["ds"], forecast_df["yhat"], label="Dự báo (yhat)")
            plt.fill_between(forecast_df["ds"], forecast_df["yhat_lower"], forecast_df["yhat_upper"], alpha=0.2, label="Khoảng tin cậy")
            if forecast_df["y"].notna().any():
                plt.plot(forecast_df["ds"], forecast_df["y"], label="Thực tế (y)", marker="o", markersize=3)
            plt.xlabel("Ngày")
            plt.ylabel("Số lượng hàng hoàn")
            plt.title("Biểu đồ dự báo")
            plt.legend()
            plt.tight_layout()
            plt.show()

            display(forecast_df.tail(10))  # hiển thị 10 dòng cuối
        except Exception as e:
            print("❌ Lỗi khi dự báo:", e)

# Gán sự kiện
run_button.on_click(on_run_clicked)

# === Hiển thị giao diện ===
controls = widgets.VBox([file_input, freq_dropdown, period_slider, run_button])
ui = widgets.VBox([controls, output_area], layout=widgets.Layout(align_items='center'))
display(ui)


In [1]:
import ipywidgets as widgets
widgets.IntSlider()

IntSlider(value=0)

In [5]:
# --- Hàm load & chuẩn hóa ---
def load_and_prepare(path=INPUT_FILE):
    df = pd.read_csv(path)
    if "Return_Date" not in df.columns or "Return_Status" not in df.columns:
        raise ValueError("CSV phải có cột 'Return_Date' và 'Return_Status'.")
    df["Return_Date"] = pd.to_datetime(df["Return_Date"], errors="coerce")
    returns = df[df["Return_Status"].astype(str).str.lower() == "returned"].copy()
    daily_returns = (
        returns.groupby("Return_Date")["Order_ID"]
        .count()
        .reset_index()
        .rename(columns={"Return_Date": "ds", "Order_ID": "y"})
    )
    full = pd.DataFrame({"ds": pd.date_range(daily_returns["ds"].min(), daily_returns["ds"].max(), freq=FREQ)})
    daily_returns = full.merge(daily_returns, on="ds", how="left")
    daily_returns["y"] = daily_returns["y"].fillna(0)
    return daily_returns


# fit Prophet
m = Prophet(daily_seasonality=True, weekly_seasonality=True, yearly_seasonality=True)
m.fit(train_df)

# forecast
future_periods = max(30, test_days)
future = m.make_future_dataframe(periods=future_periods, freq=FREQ)
forecast = m.predict(future)


# Evaluate on test set
eval_test = eval_df[eval_df["ds"].isin(test_df["ds"])].dropna(subset=["y","yhat"])
mae = mean_absolute_error(eval_test["y"], eval_test["yhat"])
try:
    rmse = mean_squared_error(eval_test["y"], eval_test["yhat"], squared=False)
except TypeError:
    mse = mean_squared_error(eval_test["y"], eval_test["yhat"])
    rmse = math.sqrt(mse)
print(f"Evaluation on holdout: MAE = {mae:.4f}, RMSE = {rmse:.4f}")



NameError: name 'INPUT_FILE' is not defined